In [1]:
!pip -q install azure-ai-documentintelligence azure-core
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 14.7 MB/s eta 0:00:00


In [2]:
"""
Batch OCR of all documents in an Azure Blob container
using Azure AI Document Intelligence (Prebuilt Read)
"""

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.storage.blob import ContainerClient
import numpy as np

# -------------------------
# Secrets (Kaggle)
# -------------------------
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

endpoint = user_secrets.get_secret("AZURE_DOCINT_ENDPOINT")
key = user_secrets.get_secret("AZURE_DOCINT_KEY")
container_sas_url = user_secrets.get_secret("AZURE_BLOB_CONTAINER_SAS_URL")

print("Endpoint loaded:", bool(endpoint))
print("Key loaded:", bool(key))
print("Container SAS loaded:", bool(container_sas_url))

# -------------------------
# Clients
# -------------------------
docint_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

container_client = ContainerClient.from_container_url(
    container_sas_url
)

# -------------------------
# Helpers
# -------------------------
def format_bounding_box(bounding_box):
    if not bounding_box:
        return "N/A"
    reshaped = np.array(bounding_box).reshape(-1, 2)
    return ", ".join(f"[{x}, {y}]" for x, y in reshaped)

# -------------------------
# Analyze all blobs
# -------------------------
print(container_client.list_blobs())
def analyze_all_documents():
    for blob in container_client.list_blobs():
        print(blob.name)

        # Skip non-doc files if needed
        if not blob.name.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
            continue

        base_url, sas = container_sas_url.split("?", 1)
        blob_url = f"{base_url}/{blob.name}?{sas}"
        print(f"\n===== Processing: {blob.name} =====")

        poller = docint_client.begin_analyze_document(
            "prebuilt-read",
            AnalyzeDocumentRequest(url_source=blob_url)
        )

        result = poller.result()

        print("Document content preview:")
        print(result.content[:500], "...\n")

        for page in result.pages:
            print(f"--- Page {page.page_number} ---")
            print(
                f"Size: {page.width} x {page.height} ({page.unit})"
            )

            for line_idx, line in enumerate(page.lines):
                print(
                    f"Line {line_idx}: '{line.content}' "
                    f"Box: {format_bounding_box(line.polygon)}"
                )

            for word in page.words:
                print(
                    f"Word '{word.content}' "
                    f"(confidence: {word.confidence})"
                )

# -------------------------
# Entry point
# -------------------------
if __name__ == "__main__":
    analyze_all_documents()

Endpoint loaded: True
Key loaded: True
Container SAS loaded: True
<iterator object azure.core.paging.ItemPaged at 0x7f48160b4bc0>
Amazon/Amazon_Invoice_1.pdf

===== Processing: Amazon/Amazon_Invoice_1.pdf =====
Document content preview:
amazon.com
Customer: John R. Doe Order number: 112-8457392-1938475 Order placed: January 18, 2026
Item
Qty
Price
USB-C Charging Cable
2
$18.99
Wireless Mouse
Subtotal: $62.48 Tax: $5.00 Order Total: $67.48
1
$24.50 ...

--- Page 1 ---
Size: 8.5 x 11.0 (LengthUnit.INCH)
Line 0: 'amazon.com' Box: [3.4812, 1.1554], [5.0045, 1.1506], [5.0045, 1.3655], [3.4812, 1.3702]
Line 1: 'Customer: John R. Doe' Box: [1.0649, 1.4848], [2.5978, 1.48], [2.5978, 1.6328], [1.0649, 1.6376]
Line 2: 'Order number: 112-8457392-1938475' Box: [1.0649, 1.6471], [3.5146, 1.6471], [3.5146, 1.7951], [1.0649, 1.7999]
Line 3: 'Order placed: January 18, 2026' Box: [1.0649, 1.8142], [3.1087, 1.8142], [3.1087, 1.9622], [1.0649, 1.967]
Line 4: 'Item' Box: [0.9885, 2.0339], [1.2798, 2.0386]